### Data analysis, cleaning and preprocessing

#### Load and analyse the data

In [3]:
import pandas as pd
df = pd.read_csv("../data/raw/characters.csv")
df.head()

,name,full_name,description,categories
0,"Abbot, Hannah","Abbot, Hannah",A Hufflepuff girl in Harry’s year and a prefec...,"['Hufflepuff', 'Dumbledore’s Army']"
1,"Abercrombie, Euan","Abercrombie, Euan",A Gryffindor student who first came to Hogwart...,['Gryffindor']
2,"Ackerley, Steward","Ackerley, Steward",A Ravenclaw student who was first introduced i...,['Ravenclaw']
3,"Aesalon, Falcon","Aesalon, Falcon","From ancient Greece, he was the first ever rec...",['Chocolate Frog Card']
4,Agnes,Agnes,A patient in St. Mungo’s who’s head is covered...,[]


In [4]:
# Remove null values
df.isna().sum()

name           39
full_name      39
description     1
categories      0
dtype: int64

In [5]:
len(df)

697

In [6]:
df[df["name"] .isna()]

,name,full_name,description,categories
363,NaN,NaN,Mangorian- A centaur who lives in the Forbidde...,['Chocolate Frog Card']
383,NaN,NaN,Merlin- Found on a wizard card and the most fa...,['Chocolate Frog Card']
385,NaN,NaN,Merwyn the Malicious- Known for creating jinxe...,['Chocolate Frog Card']
390,NaN,NaN,Moaning Myrtle- A ghost of a Hogwarts student ...,['Ravenclaw']
392,NaN,NaN,Montague- Chaser and Captain for the Slytherin...,"['quidditch', 'Slytherin']"
396,NaN,NaN,"Moony- Remus Lupin’s (see Lupin, Remus J.) nic...","['Order of the Phoenix', 'Hogwarts Staff', 'Gr..."
398,NaN,NaN,Mopsus- Defeated the Seer Calchas.,['Chocolate Frog Card']
399,NaN,NaN,Moran- Chaser for the Irish National Quidditch...,['quidditch']
401,NaN,NaN,"Morgana- See Le Fe, Morgana",['Chocolate Frog Card']
402,NaN,NaN,Morholt- Giant brother of the King of Ireland.,['Chocolate Frog Card']


In [7]:
df[df["full_name"] .isna()]

,name,full_name,description,categories
363,NaN,NaN,Mangorian- A centaur who lives in the Forbidde...,['Chocolate Frog Card']
383,NaN,NaN,Merlin- Found on a wizard card and the most fa...,['Chocolate Frog Card']
385,NaN,NaN,Merwyn the Malicious- Known for creating jinxe...,['Chocolate Frog Card']
390,NaN,NaN,Moaning Myrtle- A ghost of a Hogwarts student ...,['Ravenclaw']
392,NaN,NaN,Montague- Chaser and Captain for the Slytherin...,"['quidditch', 'Slytherin']"
396,NaN,NaN,"Moony- Remus Lupin’s (see Lupin, Remus J.) nic...","['Order of the Phoenix', 'Hogwarts Staff', 'Gr..."
398,NaN,NaN,Mopsus- Defeated the Seer Calchas.,['Chocolate Frog Card']
399,NaN,NaN,Moran- Chaser for the Irish National Quidditch...,['quidditch']
401,NaN,NaN,"Morgana- See Le Fe, Morgana",['Chocolate Frog Card']
402,NaN,NaN,Morholt- Giant brother of the King of Ireland.,['Chocolate Frog Card']


In [8]:
df[df["description"].isna()]

,name,full_name,description,categories
687,NaN,NaN,NaN,[]


We can see that a lot of the null values in the name column are due to the name being attached to the description. We will remove only the rows where all the fields are null and then fix the ones where the name can be found in the description.

In [10]:
df = df.dropna(subset=["name", "full_name", "description"], how="all")

In [11]:
len(df)

696

In [12]:
# Now let's add the names from the description if name or full_name is missing

def extract_name_from_description(row):
    if pd.isna(row["name"]) or pd.isna(row["full_name"]):
        desc = row["description"]
        if pd.notna(desc):
            head, sep, tail = desc.partition("- ")
            if sep:  # we actually found the separator
                head = head.strip()
                if pd.isna(row["name"]):
                    row["name"] = head
                if pd.isna(row["full_name"]):
                    row["full_name"] = head
                row["description"] = tail.strip()
    return row


In [13]:
df.iloc[385]

name                                                         NaN
full_name                                                    NaN
description    Merwyn the Malicious- Known for creating jinxe...
categories                               ['Chocolate Frog Card']
Name: 385, dtype: object

In [14]:
extract_name_from_description(df.iloc[385])


name                            Merwyn the Malicious
full_name                       Merwyn the Malicious
description    Known for creating jinxes and curses.
categories                   ['Chocolate Frog Card']
Name: 385, dtype: object

In [15]:
df = df.apply(extract_name_from_description, axis=1)
df.iloc[385]

name                            Merwyn the Malicious
full_name                       Merwyn the Malicious
description    Known for creating jinxes and curses.
categories                   ['Chocolate Frog Card']
Name: 385, dtype: object

In [16]:
# Check that there are no more missing names
df[df["name"] .isna()]

,name,full_name,description,categories
647,NaN,NaN,OTHER WEASLEYS,[]


In [17]:
# Let's manually remove that line
df = df.drop(index=647)

In [18]:
# Check that there are no more missing full names
df[df["full_name"].isna()]

,name,full_name,description,categories


In [19]:
# Check that there are no more null descriptions
df[df["description"].isna()]

,name,full_name,description,categories


Now that we removed the null values, let's examine the names and make sure they're formatted correctly.

In [21]:
# Let's check for non-alphanumeric characters in names
non_alphanum_filter = df["name"].str.contains(r"[^a-zA-Z0-9\s]", regex=True)
df[non_alphanum_filter]

,name,full_name,description,categories
0,"Abbot, Hannah","Abbot, Hannah",A Hufflepuff girl in Harry’s year and a prefec...,"['Hufflepuff', 'Dumbledore’s Army']"
1,"Abercrombie, Euan","Abercrombie, Euan",A Gryffindor student who first came to Hogwart...,['Gryffindor']
2,"Ackerley, Steward","Ackerley, Steward",A Ravenclaw student who was first introduced i...,['Ravenclaw']
3,"Aesalon, Falcon","Aesalon, Falcon","From ancient Greece, he was the first ever rec...",['Chocolate Frog Card']
5,"Agrippa, Cornelius","Agrippa, Cornelius",1535 and was on a wizard card saying celebrate...,['Chocolate Frog Card']
...,...,...,...,...
684,"Worple, Eldred","Worple, Eldred",An old friend of Slughorn who is the author of...,[]
688,? Yaxley,? Yaxley,A death eater who was never sent to Azkaban.,['Death Eater']
689,Lysandra- Wife of Arcturus Black who lived 188...,Lysandra- Wife of Arcturus Black who lived 188...,"1959. Had two daughters, Callidora and Charis;...",[]
693,"Zabini, ?","Zabini, ?",Was famous for being beautiful and marrying 7 ...,[]


In [22]:
non_alphanum_filter.sum()

361

In [23]:
# grab all non-alphanum chars per row (NaNs -> empty string)
chars = (
    df["full_name"]
    .str.findall(r"[^a-zA-Z0-9\s]")
    .explode()
    .dropna()
    .unique()
)

print(chars)

[',' '(' '’' ')' '.' '?' '“' '”' '-']


In [24]:
df[non_alphanum_filter]["full_name"].unique(), non_alphanum_filter.sum()

(array(['Abbot, Hannah', 'Abercrombie, Euan', 'Ackerley, Steward',
        'Aesalon, Falcon', 'Agrippa, Cornelius', 'Alderton, Archibald',
        'Aubrey, Bertram', 'Baddock, Malcolm', 'Bagman, Ludo',
        'Bagman, Otto', 'Bagman, (Ludo and Otto’s father)',
        'Bagnold, Millicent', 'Bagshot, Bathilda', 'Barbary, Heathcote',
        'Barkwith, Musidora', 'Bashir, Ali', 'Bayliss, Hetty',
        'Beamish, Oswald', 'Belby, Flavius', 'Belby, Marcus',
        'Belcher, Humphrey', 'Bell, Katie', 'Benson, Amy',
        'Bishop, Dennis', '(Black) Andromeda', '(Black) Araminta Meliflua',
        '(Black) Bellatrix Lestrange', '(Black) Aunt Elladora',
        '(Black) Narcissa', 'Black, Regulus', 'Black, Sirius',
        '(Black) Uncle Alfred', 'Blane, Balfour', 'Bletchley, Miles',
        'Blott. ?s', 'Bloxam, Beatrix', 'Bludd, Blodwyn',
        'Boardman, Stubby', 'Bode, Broderick', 'Bonnacord, Pierre',
        'Bones, Amelia', 'Bones, Edgar',
        'Bones, ? (Grandmother of Susan B

There are several problems:
1) Description included in the name (includes "-")
2) Question marks and extra spaces
3) The "Black" and "Weasly" surnames in parenthesis instead of preceding the name with a comma like the rest of the df
4) Nicknames in parenthesis
5) Quote marks in the names with descriptions

Let's examine each case individually and clean the data for each issue independently.

In [26]:
# Check that the rows where the name has a description doesn't have an empty description
df[df["full_name"].str.contains("-")]

,name,full_name,description,categories
135,"Delaney- Podmore, Patrick","Delaney- Podmore, Patrick",(Sir Patrick Delaney-Podmore) Leader of the He...,[]
181,Ethelred the Ever-Ready,Ethelred the Ever-Ready,Wizard who was famous for being offended easil...,['Chocolate Frog Card']
199,"Finch-Fletchley, Justin","Finch-Fletchley, Justin",A muggle born Hufflepuff boy in Harry’s year. ...,"['Hufflepuff', 'Dumbledore’s Army']"
249,"Grubbly-Plank, Professor","Grubbly-Plank, Professor",A witch who temporarily substitutes for Hagrid...,['Hogwarts Staff']
357,Draco - The son of Lucius and Narcissa Malfoy ...,Draco - The son of Lucius and Narcissa Malfoy ...,the Blacks and the Malfoys. He does not think ...,"['quidditch', 'Slytherin', 'Related to the Bla..."
388,Sir Nicholas de Mimsy-Porpington,Sir Nicholas de Mimsy-Porpington,The full name of the Gryffindor House Ghost (m...,['Gryffindor']
394,Alastor- Nicknamed Mad Moody,Alastor- Nicknamed Mad Moody,Eye Moody. A retired auror from the ministry o...,"['Order of the Phoenix', 'Hogwarts Staff', 'Mi..."
419,Phineas- The great Nigellus,Phineas- The great Nigellus,great grandfather of Sirius Black who was (as ...,"['Hogwarts Staff', 'Slytherin', 'Related to th..."
423,Theodore- A “weedy Nott,Theodore- A “weedy Nott,looking” student who hangs out with Draco Malf...,['Slytherin']
450,Jimmy- A boy described as short but broad Peakes,Jimmy- A boy described as short but broad Peakes,chested and in his third year. He made the Gry...,"['quidditch', 'Gryffindor']"


In [27]:
# Some of the instances are nicknames, not descriptions, so let's filter those out and just work on the descriptions in this step
nickname_indexes = [135, 181, 199, 249, 388, 394, 459]

In [28]:
# 1 Remove descriptions from names
def remove_description_from_name(row):
    name = row["full_name"]
    if pd.notna(name) and row.name not in nickname_indexes:
        head, sep, tail = name.partition("- ")
        if sep:  # we actually found the separator
            row["full_name"] = head.strip()
        row["description"] = tail.strip()
    return row

df = df.apply(remove_description_from_name, axis=1)
df[non_alphanum_filter]["full_name"].unique(), df[non_alphanum_filter].shape[0]

(array(['Abbot, Hannah', 'Abercrombie, Euan', 'Ackerley, Steward',
        'Aesalon, Falcon', 'Agrippa, Cornelius', 'Alderton, Archibald',
        'Aubrey, Bertram', 'Baddock, Malcolm', 'Bagman, Ludo',
        'Bagman, Otto', 'Bagman, (Ludo and Otto’s father)',
        'Bagnold, Millicent', 'Bagshot, Bathilda', 'Barbary, Heathcote',
        'Barkwith, Musidora', 'Bashir, Ali', 'Bayliss, Hetty',
        'Beamish, Oswald', 'Belby, Flavius', 'Belby, Marcus',
        'Belcher, Humphrey', 'Bell, Katie', 'Benson, Amy',
        'Bishop, Dennis', '(Black) Andromeda', '(Black) Araminta Meliflua',
        '(Black) Bellatrix Lestrange', '(Black) Aunt Elladora',
        '(Black) Narcissa', 'Black, Regulus', 'Black, Sirius',
        '(Black) Uncle Alfred', 'Blane, Balfour', 'Bletchley, Miles',
        'Blott. ?s', 'Bloxam, Beatrix', 'Bludd, Blodwyn',
        'Boardman, Stubby', 'Bode, Broderick', 'Bonnacord, Pierre',
        'Bones, Amelia', 'Bones, Edgar',
        'Bones, ? (Grandmother of Susan B

In [29]:
# 2 Remove question marks and extra spaces
def clean_question_marks_and_spaces(row):
    name = row["full_name"]
    if pd.notna(name):
        # Remove question marks
        name = name.replace("?", "")
        # Remove extra spaces
        name = " ".join(name.split())
        row["full_name"] = name
    return row

df = df.apply(clean_question_marks_and_spaces, axis=1)  
df[non_alphanum_filter]["full_name"].unique(), df[non_alphanum_filter].shape[0]

(array(['Abbot, Hannah', 'Abercrombie, Euan', 'Ackerley, Steward',
        'Aesalon, Falcon', 'Agrippa, Cornelius', 'Alderton, Archibald',
        'Aubrey, Bertram', 'Baddock, Malcolm', 'Bagman, Ludo',
        'Bagman, Otto', 'Bagman, (Ludo and Otto’s father)',
        'Bagnold, Millicent', 'Bagshot, Bathilda', 'Barbary, Heathcote',
        'Barkwith, Musidora', 'Bashir, Ali', 'Bayliss, Hetty',
        'Beamish, Oswald', 'Belby, Flavius', 'Belby, Marcus',
        'Belcher, Humphrey', 'Bell, Katie', 'Benson, Amy',
        'Bishop, Dennis', '(Black) Andromeda', '(Black) Araminta Meliflua',
        '(Black) Bellatrix Lestrange', '(Black) Aunt Elladora',
        '(Black) Narcissa', 'Black, Regulus', 'Black, Sirius',
        '(Black) Uncle Alfred', 'Blane, Balfour', 'Bletchley, Miles',
        'Blott. s', 'Bloxam, Beatrix', 'Bludd, Blodwyn',
        'Boardman, Stubby', 'Bode, Broderick', 'Bonnacord, Pierre',
        'Bones, Amelia', 'Bones, Edgar',
        'Bones, (Grandmother of Susan Bone

In [30]:
df[df["full_name"].str.contains("-")]

,name,full_name,description,categories
135,"Delaney- Podmore, Patrick","Delaney- Podmore, Patrick",(Sir Patrick Delaney-Podmore) Leader of the He...,[]
181,Ethelred the Ever-Ready,Ethelred the Ever-Ready,Wizard who was famous for being offended easil...,['Chocolate Frog Card']
199,"Finch-Fletchley, Justin","Finch-Fletchley, Justin",A muggle born Hufflepuff boy in Harry’s year. ...,"['Hufflepuff', 'Dumbledore’s Army']"
249,"Grubbly-Plank, Professor","Grubbly-Plank, Professor",A witch who temporarily substitutes for Hagrid...,['Hogwarts Staff']
388,Sir Nicholas de Mimsy-Porpington,Sir Nicholas de Mimsy-Porpington,The full name of the Gryffindor House Ghost (m...,['Gryffindor']
394,Alastor- Nicknamed Mad Moody,Alastor- Nicknamed Mad Moody,Eye Moody. A retired auror from the ministry o...,"['Order of the Phoenix', 'Hogwarts Staff', 'Mi..."
459,"Perks, Sally-Ann","Perks, Sally-Ann",A student in Harry’s year who was sorted befor...,[]


In [31]:
# 3 Extract surnames from parenthesis and place them before the name with a comma
df[df["full_name"].str.contains(r"\(.*\)")]

,name,full_name,description,categories
18,"Bagman, (Ludo and Otto’s father)","Bagman, (Ludo and Otto’s father)",,[]
43,(Black) Andromeda,(Black) Andromeda,,['Related to the Blacks']
44,(Black) Araminta Meliflua,(Black) Araminta Meliflua,,['Related to the Blacks']
45,(Black) Bellatrix Lestrange,(Black) Bellatrix Lestrange,,"['Slytherin', 'Death Eater', 'Related to the B..."
46,(Black) Aunt Elladora,(Black) Aunt Elladora,,"['Slytherin', 'Related to the Blacks']"
47,(Black) Narcissa,(Black) Narcissa,,"['Slytherin', 'Death Eater', 'Related to the B..."
50,(Black) Uncle Alfred,(Black) Uncle Alfred,,['Related to the Blacks']
64,"Bones, ? (Grandmother of Susan Bones)","Bones, (Grandmother of Susan Bones)",,[]
167,"Dursley, Marjorie (Marge)","Dursley, Marjorie (Marge)",,[]
212,Flourish and Blotts (Manager),Flourish and Blotts (Manager),,[]


In [32]:
# There are more cases of parenthesis, so let's make a replacement dict for the general cases and then examine the exceptions individually
import re
replacement_dict = {
    r"\(Black\)": "Black, ",
    r"\(Weasly\)": "Weasly, ",
    r"\(Granger\)": "Granger, ",
    r"\(Weasly\)\,": "Weasly, ",
    r"\(Delacour\)": "Delacour, ",
    r"\(\)": "",
}

def replace_parenthesis_surnames(row):
    name = row["full_name"]
    if pd.notna(name):
        for pattern, replacement in replacement_dict.items():
            name = re.sub(pattern, replacement, name)
        row["full_name"] = name
    return row

df = df.apply(replace_parenthesis_surnames, axis=1) 
df[non_alphanum_filter]["full_name"].unique(), df[non_alphanum_filter].shape[0]

(array(['Abbot, Hannah', 'Abercrombie, Euan', 'Ackerley, Steward',
        'Aesalon, Falcon', 'Agrippa, Cornelius', 'Alderton, Archibald',
        'Aubrey, Bertram', 'Baddock, Malcolm', 'Bagman, Ludo',
        'Bagman, Otto', 'Bagman, (Ludo and Otto’s father)',
        'Bagnold, Millicent', 'Bagshot, Bathilda', 'Barbary, Heathcote',
        'Barkwith, Musidora', 'Bashir, Ali', 'Bayliss, Hetty',
        'Beamish, Oswald', 'Belby, Flavius', 'Belby, Marcus',
        'Belcher, Humphrey', 'Bell, Katie', 'Benson, Amy',
        'Bishop, Dennis', 'Black,  Andromeda', 'Black,  Araminta Meliflua',
        'Black,  Bellatrix Lestrange', 'Black,  Aunt Elladora',
        'Black,  Narcissa', 'Black, Regulus', 'Black, Sirius',
        'Black,  Uncle Alfred', 'Blane, Balfour', 'Bletchley, Miles',
        'Blott. s', 'Bloxam, Beatrix', 'Bludd, Blodwyn',
        'Boardman, Stubby', 'Bode, Broderick', 'Bonnacord, Pierre',
        'Bones, Amelia', 'Bones, Edgar',
        'Bones, (Grandmother of Susan Bone

In [33]:
df.iloc[43]

name                   (Black) Andromeda
full_name              Black,  Andromeda
description                             
categories     ['Related to the Blacks']
Name: 43, dtype: object

In [34]:
df[df["full_name"].str.contains(r"\(.*\)")]

,name,full_name,description,categories
18,"Bagman, (Ludo and Otto’s father)","Bagman, (Ludo and Otto’s father)",,[]
64,"Bones, ? (Grandmother of Susan Bones)","Bones, (Grandmother of Susan Bones)",,[]
167,"Dursley, Marjorie (Marge)","Dursley, Marjorie (Marge)",,[]
212,Flourish and Blotts (Manager),Flourish and Blotts (Manager),,[]
226,(Guant) Tom Marvolo,(Guant) Tom Marvolo,,"['Slytherin', 'Death Eater']"
227,Ghoul (of the Weasleys),Ghoul (of the Weasleys),,[]
228,Giant Squid (of Hogwarts),Giant Squid (of Hogwarts),,[]
242,Grey Lady (Helena Ravenclaw),Grey Lady (Helena Ravenclaw),,['Ravenclaw']
341,Nymphadora (Dora) Tonks Lupin,Nymphadora (Dora) Tonks Lupin,,[]
412,Muriel (Aunt),Muriel (Aunt),,[]


In [35]:
# Let's handle the descriptions that are still in parenthesis
def handle_descriptions_in_parenthesis(row):
    name = row["full_name"]
    if pd.notna(name):
        head, sep, tail = name.partition(" (")
        if sep:  # we actually found the separator
            description = tail.rstrip(")")
            row["full_name"] = head.strip()
            if pd.isna(row["description"]) or row["description"] == "":
                row["description"] = description.strip()
    return row

desc_indexes = [18, 64, 212]
df.iloc[desc_indexes] = df.iloc[desc_indexes].apply(handle_descriptions_in_parenthesis, axis=1)



In [36]:
df[df["full_name"].str.contains(r"\(.*\)")]

,name,full_name,description,categories
167,"Dursley, Marjorie (Marge)","Dursley, Marjorie (Marge)",,[]
226,(Guant) Tom Marvolo,(Guant) Tom Marvolo,,"['Slytherin', 'Death Eater']"
227,Ghoul (of the Weasleys),Ghoul (of the Weasleys),,[]
228,Giant Squid (of Hogwarts),Giant Squid (of Hogwarts),,[]
242,Grey Lady (Helena Ravenclaw),Grey Lady (Helena Ravenclaw),,['Ravenclaw']
341,Nymphadora (Dora) Tonks Lupin,Nymphadora (Dora) Tonks Lupin,,[]
412,Muriel (Aunt),Muriel (Aunt),,[]
514,Helena (The Grey Lady) Ravenclaw,Helena (The Grey Lady) Ravenclaw,,['Ravenclaw']
535,Newton (Newt) Artemis Fido Scamander,Newton (Newt) Artemis Fido Scamander,,"['Chocolate Frog Card', 'Ministry of Magic']"
643,Ginevra (Ginny) Weasley,Ginevra (Ginny) Weasley,,"['Quidditch Player', 'gryff', 'Slug Club', 'Re..."


In [37]:
# Separate the nicknames from the full names with commas
def extract_nickname(row):
    full_name = row.get("full_name")
    if not isinstance(full_name, str) or not full_name:
        return row

    matches = re.findall(r"\(([^)]+)\)", full_name)
    if not matches:
        return row

    nickname = ", ".join(matches)
    cleaned = re.sub(r"\s*\(([^)]+)\)", "", full_name)
    cleaned = " ".join(cleaned.split())

    row["full_name"] = f"{cleaned}, {nickname}"
    row["nickname"] = nickname  # optional new column

    return row

extract_nickname(df.iloc[643])

name                                     Ginevra (Ginny) Weasley
full_name                                 Ginevra Weasley, Ginny
description                                                     
categories     ['Quidditch Player', 'gryff', 'Slug Club', 'Re...
nickname                                                   Ginny
Name: 643, dtype: object

In [38]:
df = df.apply(extract_nickname, axis=1)

In [39]:
df[df["full_name"].str.contains(r"\(.*\)")]

,categories,description,full_name,name,nickname


In [40]:
# Remove quote marks
df[df["full_name"].str.contains(r'“|”|’')] 

,categories,description,full_name,name,nickname
80,[],,"Brook Stanton, Rupert “Axebanger”","Brook Stanton, Rupert “Axebanger”",NaN
307,[],,Kevin’s mother,Kevin’s mother,NaN
343,[],,Ted “Teddy” Remus Lupid,Ted “Teddy” Remus Lupid,NaN
431,['quidditch'],,Darren O’Hare,Darren O’Hare,NaN
634,[],Street peddler who was accused by the Ministry...,“Honest Willy”,“Honest Willy”- Street peddler who was accused...,NaN


In [41]:
# The quotes that remain are nicknames, so let's extract them
def extract_nickname_from_quotes(row):
    full_name = row.get("full_name")
    if not isinstance(full_name, str) or not full_name:
        return row

    matches = re.findall(r"\“([^”]+)\”", full_name)
    if not matches:
        return row

    nickname = ", ".join(matches)
    cleaned = re.sub(r"\s*\“[^”]+\”", "", full_name)
    cleaned = " ".join(cleaned.split())

    row["full_name"] = f"{cleaned}, {nickname}"
    row["nickname"] = nickname  # optional new column

    return row

In [42]:
df = df.apply(extract_nickname_from_quotes, axis=1)
df[df["full_name"].str.contains(r'“|”|’')] 

,categories,description,full_name,name,nickname
307,[],,Kevin’s mother,Kevin’s mother,NaN
431,['quidditch'],,Darren O’Hare,Darren O’Hare,NaN


In [43]:
df.iloc[343]

categories                          []
description                           
full_name       Ted Remus Lupid, Teddy
name           Ted “Teddy” Remus Lupid
nickname                         Teddy
Name: 343, dtype: object

In [44]:
# Let's check how the full names with special characters look now
df[non_alphanum_filter]["full_name"].unique(), non_alphanum_filter.sum()

(array(['Abbot, Hannah', 'Abercrombie, Euan', 'Ackerley, Steward',
        'Aesalon, Falcon', 'Agrippa, Cornelius', 'Alderton, Archibald',
        'Aubrey, Bertram', 'Baddock, Malcolm', 'Bagman, Ludo',
        'Bagman, Otto', 'Bagman,', 'Bagnold, Millicent',
        'Bagshot, Bathilda', 'Barbary, Heathcote', 'Barkwith, Musidora',
        'Bashir, Ali', 'Bayliss, Hetty', 'Beamish, Oswald',
        'Belby, Flavius', 'Belby, Marcus', 'Belcher, Humphrey',
        'Bell, Katie', 'Benson, Amy', 'Bishop, Dennis',
        'Black,  Andromeda', 'Black,  Araminta Meliflua',
        'Black,  Bellatrix Lestrange', 'Black,  Aunt Elladora',
        'Black,  Narcissa', 'Black, Regulus', 'Black, Sirius',
        'Black,  Uncle Alfred', 'Blane, Balfour', 'Bletchley, Miles',
        'Blott. s', 'Bloxam, Beatrix', 'Bludd, Blodwyn',
        'Boardman, Stubby', 'Bode, Broderick', 'Bonnacord, Pierre',
        'Bones, Amelia', 'Bones, Edgar', 'Bones,', 'Bones, Susan',
        'Bonham, Mungo', 'Boot, Terry', '

In [45]:
# grab all non-alphanum chars per row (NaNs -> empty string)
chars = (
    df["full_name"]
    .str.findall(r"[^a-zA-Z0-9\s]")
    .explode()
    .dropna()
    .unique()
)

print(chars)

[',' '.' '-' '’']


Now let's check how the special characters we got look.

In [47]:
df[df["full_name"].str.contains(r"\,")]["full_name"].unique()

array(['Abbot, Hannah', 'Abercrombie, Euan', 'Ackerley, Steward',
       'Aesalon, Falcon', 'Agrippa, Cornelius', 'Alderton, Archibald',
       'Aubrey, Bertram', 'Baddock, Malcolm', 'Bagman, Ludo',
       'Bagman, Otto', 'Bagman,', 'Bagnold, Millicent',
       'Bagshot, Bathilda', 'Barbary, Heathcote', 'Barkwith, Musidora',
       'Bashir, Ali', 'Bayliss, Hetty', 'Beamish, Oswald',
       'Belby, Flavius', 'Belby, Marcus', 'Belcher, Humphrey',
       'Bell, Katie', 'Benson, Amy', 'Bishop, Dennis',
       'Black,  Andromeda', 'Black,  Araminta Meliflua',
       'Black,  Bellatrix Lestrange', 'Black,  Aunt Elladora',
       'Black,  Narcissa', 'Black, Regulus', 'Black, Sirius',
       'Black,  Uncle Alfred', 'Blane, Balfour', 'Bletchley, Miles',
       'Bloxam, Beatrix', 'Bludd, Blodwyn', 'Boardman, Stubby',
       'Bode, Broderick', 'Bonnacord, Pierre', 'Bones, Amelia',
       'Bones, Edgar', 'Bones,', 'Bones, Susan', 'Bonham, Mungo',
       'Boot, Terry', 'Boothby, Gladys', 'Borage, L

In [48]:
# Let's remove the commans that are not separating surnames or nicknames
df["full_name"] = (
    df["full_name"]
    .str.replace(r"^\s*,\s*", "", regex=True)   # leading commas/spaces
    .str.replace(r"\s*,\s*$", "", regex=True)   # trailing commas/spaces
    .str.replace(r"\s*,\s*", ", ", regex=True)  # collapse inner whitespace
    .str.replace(r"\s+", " ", regex=True)       # collapse spaces in general
    .str.strip()
)

df[df["full_name"].str.contains(r"\,")]["full_name"].unique()


array(['Abbot, Hannah', 'Abercrombie, Euan', 'Ackerley, Steward',
       'Aesalon, Falcon', 'Agrippa, Cornelius', 'Alderton, Archibald',
       'Aubrey, Bertram', 'Baddock, Malcolm', 'Bagman, Ludo',
       'Bagman, Otto', 'Bagnold, Millicent', 'Bagshot, Bathilda',
       'Barbary, Heathcote', 'Barkwith, Musidora', 'Bashir, Ali',
       'Bayliss, Hetty', 'Beamish, Oswald', 'Belby, Flavius',
       'Belby, Marcus', 'Belcher, Humphrey', 'Bell, Katie', 'Benson, Amy',
       'Bishop, Dennis', 'Black, Andromeda', 'Black, Araminta Meliflua',
       'Black, Bellatrix Lestrange', 'Black, Aunt Elladora',
       'Black, Narcissa', 'Black, Regulus', 'Black, Sirius',
       'Black, Uncle Alfred', 'Blane, Balfour', 'Bletchley, Miles',
       'Bloxam, Beatrix', 'Bludd, Blodwyn', 'Boardman, Stubby',
       'Bode, Broderick', 'Bonnacord, Pierre', 'Bones, Amelia',
       'Bones, Edgar', 'Bones, Susan', 'Bonham, Mungo', 'Boot, Terry',
       'Boothby, Gladys', 'Borage, Libatius', 'Bott, Bertie',
       '

In [49]:
df[df["full_name"].str.contains("Zabini")]

,categories,description,full_name,name,nickname
692,"['slyth', 'Slug Club']",,Blaise Zabini,Blaise Zabini,NaN
693,[],,Zabini,"Zabini, ?",NaN


In [50]:
# Now let's check the period characters
df[df["full_name"].str.contains(r"\.")]["full_name"].unique()

array(['Blott. s', 'Bunky, K.', 'Capper, S.', 'Crouch, Barty Sr.',
       'Crouch, Barty Jr.', 'Dorny, J.', 'Fawcett, S.', 'King, R.J.H.',
       'Remus J. Lupin', 'M. G. McGonagall', 'Mr. Paws', 'Norris, Mrs.',
       'Paws, Mr.', 'Prod, D. J.'], dtype=object)

They are all full names or abbreviations of the names so they look fine, let's move on to the dashes.

In [52]:
# Let's check how the dashes look
df[df["full_name"].str.contains(r"\-")]["full_name"].unique()

array(['Delaney- Podmore, Patrick', 'Ethelred the Ever-Ready',
       'Finch-Fletchley, Justin', 'Grubbly-Plank, Professor',
       'Sir Nicholas de Mimsy-Porpington', 'Alastor- Nicknamed Mad Moody',
       'Perks, Sally-Ann'], dtype=object)

Most of them are ok let's manually change the mad moody nickname 

In [54]:
df[df["full_name"].str.contains("Alastor")]

,categories,description,full_name,name,nickname
394,"['Order of the Phoenix', 'Hogwarts Staff', 'Mi...",Eye Moody. A retired auror from the ministry o...,Alastor- Nicknamed Mad Moody,Alastor- Nicknamed Mad Moody,NaN


In [55]:
df.at[394, "full_name"] = df.at[394, "full_name"].replace("- Nicknamed ", ", ")
df.at[394, "nickname"] = "Mad Moody"
df.iloc[394]

categories     ['Order of the Phoenix', 'Hogwarts Staff', 'Mi...
description    Eye Moody. A retired auror from the ministry o...
full_name                                     Alastor, Mad Moody
name                                Alastor- Nicknamed Mad Moody
nickname                                               Mad Moody
Name: 394, dtype: object

In [56]:
# Remove the commas
import re

def reorder_comma_name(s):
    # Si no hay coma → devolver igual
    if "," not in s:
        return s.strip()

    before, after = map(str.strip, s.split(",", 1))
    return f"{after} {before}".strip()

df["full_name"] = df["full_name"].apply(reorder_comma_name)
df["full_name"].unique()

array(['Hannah Abbot', 'Euan Abercrombie', 'Steward Ackerley',
       'Falcon Aesalon', 'Agnes', 'Cornelius Agrippa',
       'Archibald Alderton', 'Alguf the Awful', 'Alphard', 'Archie',
       'Aragog', 'Arnold', 'Bertram Aubrey', 'Avery', 'Babayaga',
       'Malcolm Baddock', 'Ludo Bagman', 'Otto Bagman', 'Bagman',
       'Millicent Bagnold', 'Bathilda Bagshot', 'Bandon Banshee', 'Bane',
       'Heathcote Barbary', 'Barker', 'Musidora Barkwith',
       'Barnabas the Barmy', 'Barny', 'Barrufio', 'Ali Bashir', 'Basil',
       'Hetty Bayliss', 'Oswald Beamish', 'Beedle the Bard',
       'Flavius Belby', 'Marcus Belby', 'Humphrey Belcher', 'Katie Bell',
       'Amy Benson', 'Billy', 'Binky', 'Binns', 'Dennis Bishop',
       'Andromeda Black', 'Araminta Meliflua Black',
       'Bellatrix Lestrange Black', 'Aunt Elladora Black',
       'Narcissa Black', 'Regulus Black', 'Sirius Black',
       'Uncle Alfred Black', 'Balfour Blane', 'Miles Bletchley',
       'Bloody Barron', 'Blott. s', 'Bea

In [57]:
# Now let's split it and check the ones that have more than 2 parts to analyse further
name_parts = df["full_name"].str.split()
name_parts_counts = name_parts.apply(len)
name_parts_counts.value_counts()    

full_name
2    428
1    192
3     58
4     12
5      5
Name: count, dtype: int64

In [58]:
len(name_parts[name_parts_counts > 2])

75

In [59]:
long_names = name_parts[name_parts_counts > 2].reset_index()
long_names

,index,full_name
0,7,"[Alguf, the, Awful]"
1,26,"[Barnabas, the, Barmy]"
2,33,"[Beedle, the, Bard]"
3,44,"[Araminta, Meliflua, Black]"
4,45,"[Bellatrix, Lestrange, Black]"
...,...,...
70,651,"[Hermione, Granger, Weasley]"
71,655,"[Weasley, Second, Cousin]"
72,660,"[Wendelin, the, Weird]"
73,671,"[Wilfred, the, Wistful]"


In [60]:
# Let's inspect the names that don't contain "the"
mask = long_names["full_name"].apply(lambda parts: "the" in parts if isinstance(parts, list) else
False)

long_names[~mask]  # rows without "the"

,index,full_name
3,44,"[Araminta, Meliflua, Black]"
4,45,"[Bellatrix, Lestrange, Black]"
5,46,"[Aunt, Elladora, Black]"
6,50,"[Uncle, Alfred, Black]"
7,76,"[Maximus, Brankovitch, III]"
8,78,"[Kevin, and, Karl, Broadmoor]"
9,80,"[Rupert,, Axebanger, Brook, Stanton]"
10,86,"[Rosalind, Antigone, Bungs]"
11,120,"[Barty, Sr., Crouch]"
12,121,"[Barty, Jr., Crouch]"


Some of the names are long but they all look fine.

#### Fix the "name" column

In [63]:
df["full_name"].unique()

array(['Hannah Abbot', 'Euan Abercrombie', 'Steward Ackerley',
       'Falcon Aesalon', 'Agnes', 'Cornelius Agrippa',
       'Archibald Alderton', 'Alguf the Awful', 'Alphard', 'Archie',
       'Aragog', 'Arnold', 'Bertram Aubrey', 'Avery', 'Babayaga',
       'Malcolm Baddock', 'Ludo Bagman', 'Otto Bagman', 'Bagman',
       'Millicent Bagnold', 'Bathilda Bagshot', 'Bandon Banshee', 'Bane',
       'Heathcote Barbary', 'Barker', 'Musidora Barkwith',
       'Barnabas the Barmy', 'Barny', 'Barrufio', 'Ali Bashir', 'Basil',
       'Hetty Bayliss', 'Oswald Beamish', 'Beedle the Bard',
       'Flavius Belby', 'Marcus Belby', 'Humphrey Belcher', 'Katie Bell',
       'Amy Benson', 'Billy', 'Binky', 'Binns', 'Dennis Bishop',
       'Andromeda Black', 'Araminta Meliflua Black',
       'Bellatrix Lestrange Black', 'Aunt Elladora Black',
       'Narcissa Black', 'Regulus Black', 'Sirius Black',
       'Uncle Alfred Black', 'Balfour Blane', 'Miles Bletchley',
       'Bloody Barron', 'Blott. s', 'Bea

In [64]:
df["name"] = df["full_name"].str.split(" ")
df["name"]

0           [Hannah, Abbot]
1       [Euan, Abercrombie]
2       [Steward, Ackerley]
3         [Falcon, Aesalon]
4                   [Agnes]
               ...         
692        [Blaise, Zabini]
693                [Zabini]
694    [Ladislaw, Zamojski]
695          [Rose, Zeller]
696                [Zograf]
Name: name, Length: 695, dtype: object

In [65]:
# ok now let's check for names that are repeated - we need a unique identifier for each character
# 1. Explode the lists so each word becomes its own row
df_exploded = df["name"].explode()

# 2. Find words that appear more than once in the entire column
repeated_words = df_exploded.value_counts()
repeated_words = repeated_words[repeated_words > 1].index

# 3. For each row, store the list of repeated words it contains
df["repeated_words"] = df["name"].apply(
    lambda lst: [word for word in lst if word in repeated_words]
)

# 4. Boolean column: does the row contain any repeated words?
df["has_repeated"] = df["repeated_words"].apply(lambda x: len(x) > 0)


In [66]:
df[df["has_repeated"]==True][["has_repeated", "repeated_words"]]

,has_repeated,repeated_words
5,True,[Cornelius]
7,True,"[the, Awful]"
11,True,[Arnold]
15,True,[Malcolm]
16,True,[Bagman]
...,...,...
680,True,[Lord]
682,True,[Augustus]
692,True,[Zabini]
693,True,[Zabini]


In [67]:
repeated_word_counts = df["repeated_words"].value_counts().reset_index()
repeated_word_counts 

,repeated_words,count
0,[],404
1,[the],13
2,[Weasley],12
3,[Longbottom],5
4,[Madam],4
...,...,...
177,"[Gwenog, Jones]",1
178,"[Patrick, Podmore]",1
179,[Higgs],1
180,"[Bertie, Higgs]",1


In [68]:
repeated_word_list = repeated_word_counts["repeated_words"].tolist()
print(repeated_word_list)


[[], ['the'], ['Weasley'], ['Longbottom'], ['Madam'], ['Scamander'], ['Lovegood'], ['Krum'], ['Delacour'], ['Bones'], ['Malfoy'], ['Diggory'], ['Augustus'], ['Smethwyk'], ['Dursley'], ['Dumbledore'], ['Bagman'], ['Gudgeon'], ['Finnigan'], ['Lestrange'], ['Morgan'], ['Montague'], ['Gaunt'], ['Professor'], ['of', 'the'], ['Kevin'], ['Cornelius'], ['Hengist', 'of'], ['Hagrid'], ['Ernie'], ['Dorcas'], ['McClivert'], ['McCormack'], ['Zabini'], ['Gilbert'], ['Willy'], ['Lord'], ['Stebbins'], ['Scrimgeour'], ['Rosier'], ['Urquhart'], ['Lily', 'Potter'], ['James', 'Potter'], ['Roddy', 'Pontner'], ['Patil'], ['Ollerton'], ['Mr.', 'Paws'], ['Myrtle'], ['Sir', 'Nicholas', 'de'], ['Elladora'], ['McLaggen'], ['McGonagall'], ['Emeric', 'the', 'Evil'], ['Hornby'], ['Edgar'], ['Daisy'], ['Black'], ['Crabbe'], ['Narcissa', 'Black'], ['J.'], ['Elfrinda'], ['Agatha'], ['Doris'], ['Carrow'], ['Basil'], ['Millicent'], ['Malcolm'], ['S.'], ['Derwent'], ['Dennis'], ['Barty', 'Crouch'], ['the', 'Awful'], ['Ha

In [69]:
df.head()

,categories,description,full_name,name,nickname,repeated_words,has_repeated
0,"['Hufflepuff', 'Dumbledore’s Army']",,Hannah Abbot,"[Hannah, Abbot]",NaN,[],False
1,['Gryffindor'],,Euan Abercrombie,"[Euan, Abercrombie]",NaN,[],False
2,['Ravenclaw'],,Steward Ackerley,"[Steward, Ackerley]",NaN,[],False
3,['Chocolate Frog Card'],,Falcon Aesalon,"[Falcon, Aesalon]",NaN,[],False
4,[],,Agnes,[Agnes],NaN,[],False


In [70]:
df.tail()

,categories,description,full_name,name,nickname,repeated_words,has_repeated
692,"['slyth', 'Slug Club']",,Blaise Zabini,"[Blaise, Zabini]",NaN,[Zabini],True
693,[],,Zabini,[Zabini],NaN,[Zabini],True
694,['Quidditch Player'],,Ladislaw Zamojski,"[Ladislaw, Zamojski]",NaN,[],False
695,['Hufflepuff'],,Rose Zeller,"[Rose, Zeller]",NaN,[Rose],True
696,['Quidditch Player'],,Zograf,[Zograf],NaN,[],False


In [71]:
# Remove repeated words from names
def subtract_repeats(row):
    repeated = set(row["repeated_words"] or [])
    tokens = row["name"] if isinstance(row["name"], list) else str(row["name"]).split()
    pruned = [tok for tok in tokens if tok not in repeated]
    row["name"] = pruned if pruned else tokens  # keep original if all tokens repeat
    return row

df = df.apply(subtract_repeats, axis=1)

In [72]:
df.tail()

,categories,description,full_name,name,nickname,repeated_words,has_repeated
692,"['slyth', 'Slug Club']",,Blaise Zabini,[Blaise],NaN,[Zabini],True
693,[],,Zabini,[Zabini],NaN,[Zabini],True
694,['Quidditch Player'],,Ladislaw Zamojski,"[Ladislaw, Zamojski]",NaN,[],False
695,['Hufflepuff'],,Rose Zeller,[Zeller],NaN,[Rose],True
696,['Quidditch Player'],,Zograf,[Zograf],NaN,[],False


In [73]:
# lets check how many unique names we have now
mask = df["name"].apply(lambda tokens: isinstance(tokens, list) and len(tokens) > 1)
df[mask][["full_name", "name", "repeated_words", "has_repeated"]]

,full_name,name,repeated_words,has_repeated
0,Hannah Abbot,"[Hannah, Abbot]",[],False
1,Euan Abercrombie,"[Euan, Abercrombie]",[],False
2,Steward Ackerley,"[Steward, Ackerley]",[],False
3,Falcon Aesalon,"[Falcon, Aesalon]",[],False
6,Archibald Alderton,"[Archibald, Alderton]",[],False
...,...,...,...,...
684,Eldred Worple,"[Eldred, Worple]",[],False
685,Bowman Wright,"[Bowman, Wright]",[],False
686,Josef Wronski,"[Josef, Wronski]",[],False
690,Cyprian Youdle,"[Cyprian, Youdle]",[],False


In [74]:
df.to_csv("../data/processed/characters_cleaned.csv", index=False)

In [75]:
# TODO:
# ordenar x nombre apellido xq algunos vienen bien y otros no
# eliminar filas duplicadas
# poner el apodo mas significativo 
# quedarse con el nombre si no se repite y si no nombre y apellido
